In [ ]:
#| include: false
# Global parameters and data (ported from wdi_hybrid_analysis.ipynb, cell 0).
import pandas as pd
start_year = 1995
discount_rate = 0.05
window_length = 15

df = pd.read_csv('../data/wdi_africa_long.csv')
df = df[df['iso3'] != 'LSO'].reset_index(drop=True)  # Lesotho CPI has multiple uncorrected basket/base-year breaks (1999, 2002, 2009)
df['usd_per_lcu'] = 1 / df['exchange_rate']
df['Q'] = df['cpi'] / df['exchange_rate']   # real exchange rate, USD per local basket


# Exchange-rate regime classification, keyed by ISO3
# (copy of source/data_analysis_functions.py:31; importing that module would
# load the MFMOD extract, which this notebook does not use).
fx_regime = {
    # CFA Franc Zone - WAEMU
    'BEN': 'EUR-linked', 'BFA': 'EUR-linked', 'CIV': 'EUR-linked', 'GNB': 'EUR-linked',
    'MLI': 'EUR-linked', 'NER': 'EUR-linked', 'SEN': 'EUR-linked', 'TGO': 'EUR-linked',
    # CFA Franc Zone - CEMAC
    'CMR': 'EUR-linked', 'CAF': 'EUR-linked', 'TCD': 'EUR-linked', 'COG': 'EUR-linked',
    'GNQ': 'EUR-linked', 'GAB': 'EUR-linked',
    # Other EUR-pegged
    'COM': 'EUR-linked', 'CPV': 'EUR-linked',
    # Common Monetary Area (LSO itself is excluded from the data)
    'LSO': 'ZAR-linked', 'NAM': 'ZAR-linked', 'SWZ': 'ZAR-linked',
}

In [ ]:
#| include: false
# Charting helpers (ported from wdi_hybrid_analysis.ipynb, cell 1:
# compute_V, shading helpers, and the two-axis country_chart).
import numpy as np
import altair as alt
# Render charts to static PNG in the kernel (needs vl-convert-python, installed
# in .venv). Viewer-proof: notebook front-ends (VS Code / JupyterLab) render raw
# vegalite JSON through their own vega bundle and can inject a dark display
# theme that overrides the spec config, hiding axis/legend/title text. PNGs
# also carry straight through to Quarto PDF. For interactive tooltips while
# exploring, temporarily switch to: alt.renderers.enable('html')
alt.renderers.enable('png', scale_factor=2.0)

# Theme pinning ALL ink colors, not just the background: dark-mode viewers
# (VS Code / JupyterLab) render vega-lite with the dark theme's light text,
# so forcing only a white background yields invisible axis labels, ticks and
# legends (white-on-white). alt.theme is the non-deprecated 5.5+ API.
@alt.theme.register('paper', enable=True)
def _paper_theme():
    ink = '#333333'
    return {'config': {
        'background': 'white',
        'title': {'color': ink},
        'axis': {'labelColor': ink, 'titleColor': ink, 'tickColor': ink,
                 'domainColor': ink, 'gridColor': '#dddddd'},
        'legend': {'labelColor': ink, 'titleColor': ink},
        'header': {'labelColor': ink, 'titleColor': ink},
        'text': {'color': ink},
    }}

def compute_V(cdata, t0, T, col='Q', rho=0.05):
    win = cdata[(cdata['year'] >= t0) & (cdata['year'] <= T)].sort_values('year')
    v = win[col].to_numpy()
    g = v / v[0] - 1.0
    t = np.arange(1, len(v) + 1, dtype=float)
    dfactors = (1.0 + rho) ** (-t)
    return float(np.sum(g * dfactors))

def _shade_and_text_layers(cdata, col, hline, shades):
    layers = []
    text_rows = []
    if shades and hline is not None:
        for shade in shades:
            y_start, y_end, color = shade[0], shade[1], shade[2]
            label = shade[3] if len(shade) > 3 else None
            seg = cdata[(cdata['year'] >= y_start) & (cdata['year'] <= y_end)].copy()
            seg['hline'] = hline
            layers.append(
                alt.Chart(seg).mark_area(opacity=0.3, color=color).encode(
                    x=alt.X('year:O'),
                    y=alt.Y(f'{col}:Q'),
                    y2=alt.Y2('hline:Q')
                )
            )
            if label is not None and not seg.empty:
                mid_year = int(seg['year'].iloc[len(seg)//2])
                mid_y = (seg[col].mean() + hline) / 2
                text_rows.append({'year': mid_year, 'y': mid_y, 'text': label, 'color': color})
    return layers, text_rows

def country_chart(iso3, start_year=start_year,
                  q_hline=None, shades=None,
                  usd_hline=None, usd_shades=None,
                  usd_y_range=None, h_y_range=None):
    cdata = df[(df['iso3'] == iso3) & (df['year'] >= start_year)].dropna(subset=['usd_per_lcu', 'Q'])
    country_name = cdata['country'].iloc[0]

    legend_setting = alt.Legend(title="Series", orient="bottom")

    usd_scale = alt.Scale(domain=list(usd_y_range)) if usd_y_range is not None else alt.Undefined
    q_scale = alt.Scale(domain=list(h_y_range)) if h_y_range is not None else alt.Undefined

    # --- LEFT AXIS (usd_per_lcu) ---
    left_layers = []
    usd_shade_layers, usd_text_rows = _shade_and_text_layers(cdata, 'usd_per_lcu', usd_hline, usd_shades)
    left_layers.extend(usd_shade_layers)

    left_layers.append(
        alt.Chart(cdata).mark_line(point=True).encode(
            x=alt.X('year:O', title='Year'),
            y=alt.Y('usd_per_lcu:Q', title='USD per LCU', scale=usd_scale),
            color=alt.Color('Metric:N', legend=legend_setting),
            tooltip=['year', 'usd_per_lcu']
        ).transform_calculate(Metric="'usd_per_lcu'")
    )

    if usd_hline is not None:
        left_layers.append(
            alt.Chart(pd.DataFrame({'y':[usd_hline]})).mark_rule(color='blue', strokeDash=[4,4]).encode(
                y=alt.Y('y:Q')
            )
        )

    if usd_text_rows:
        left_layers.append(
            alt.Chart(pd.DataFrame(usd_text_rows)).mark_text(fontSize=14, fontWeight='bold').encode(
                x=alt.X('year:O'),
                y=alt.Y('y:Q'),
                text=alt.Text('text:N'),
                color=alt.Color('color:N', scale=None, legend=None)
            )
        )

    left_chart = alt.layer(*left_layers)

    # --- RIGHT AXIS (Q) ---
    right_layers = []
    q_shade_layers, q_text_rows = _shade_and_text_layers(cdata, 'Q', q_hline, shades)
    right_layers.extend(q_shade_layers)

    right_layers.append(
        alt.Chart(cdata).mark_line(point=True, color='orange').encode(
            x=alt.X('year:O', title='Year'),
            y=alt.Y('Q:Q', title='Q = CPI / S  (USD per local basket)', axis=alt.Axis(orient='right'), scale=q_scale),
            color=alt.Color('Metric:N', legend=legend_setting),
            tooltip=['year', 'Q']
        ).transform_calculate(Metric="'Q'")
    )

    if q_hline is not None:
        right_layers.append(
            alt.Chart(pd.DataFrame({'y':[q_hline]})).mark_rule(color='red', strokeDash=[4,4]).encode(
                y=alt.Y('y:Q')
            )
        )

    if q_text_rows:
        right_layers.append(
            alt.Chart(pd.DataFrame(q_text_rows)).mark_text(fontSize=14, fontWeight='bold').encode(
                x=alt.X('year:O'),
                y=alt.Y('y:Q'),
                text=alt.Text('text:N'),
                color=alt.Color('color:N', scale=None, legend=None)
            )
        )

    right_chart = alt.layer(*right_layers)

    return alt.layer(left_chart, right_chart).resolve_scale(y='independent').properties(
        title=f"{country_name} (from {start_year})",
        width=500,
        height=350
    )

dza_data = df[(df['iso3']=='DZA') & (df['year']>=start_year)].dropna(subset=['Q'])
q_1998 = dza_data.loc[dza_data['year']==1998, 'Q'].iloc[0]
usd_1998 = df[(df['iso3']=='DZA') & (df['year']==1998)]['usd_per_lcu'].iloc[0]
V_red   = compute_V(dza_data, 1998, 2006, col='Q')
V_green = compute_V(dza_data, 2006, 2013, col='Q')
V_total = V_red+V_green
V_blue  = compute_V(dza_data, 1998, 2013, col='usd_per_lcu')
print(f"Q_1998 (Algeria)         = {q_1998:.4f}")
print(f"usd_per_lcu_1998 (Algeria) = {usd_1998:.6f}")
print(f"V_Q(1998,2006)            = {V_red:.4f}")
print(f"V_Q(2006,2013)            = {V_green:.4f}")
print(f"V_Q(1998,2013)            = {V_total:.4f}")
print(f"V_1/S(1998,2013)          = {V_blue:.4f}")

country_chart(
    'DZA',
    q_hline=q_1998,
    shades=[
        (1998, 2006, 'red',   f"{V_red:.3f}"),
        (2006, 2013, 'green', f"{V_green:.3f}"),
    ],
    usd_hline=usd_1998,
    usd_shades=[
        (1998, 2013, 'blue', f"{V_blue:.3f}"),
    ],
    usd_y_range=(0.006, 0.030),
)

# Motivation: the currency mismatch that blocks the deal

Currency mismatch between revenues from the sale of electricity in developing countries and the cost of hard currency debt service holds up many infrastructure finance transactions. When such hard currency credit has been provided for developing country electricity generation infrastructure, the currency risk has often been placed on the national utility - "offtaker" - through dollarised power purchase agreements (PPAs). This has led to an exacerbation of these utilities' financial woes from currency devaluations. This has had the knock-on effect of increasing offtaker risk for such projects and generating an additional impediment to the financing of electricity generation projects. Capital costs for such projects are thus affected two-fold by the currency mismatch, by the direct currency risk and by the induced offtaker risk.

This proposal, analogous to the emergency liquidity facility of the EcoInvest Brazil platform, proposes to split the risk into an inflation risk component, to be shared by the final consumers and the utility (offtaker in the PPAs), and a real component, for which it is possible to structure affordable cover, as we shall see below.

## Notation

For country $i$, with financial close normalised to $t=0$, maturity $T$, and tenor $n=T$:

- $P_{i,t}$ — domestic price level (CPI), in LCU; $p\equiv\ln P$, inflation $\pi=\dot p$
- $S_{i,t}$ — nominal exchange rate in LCU per USD, so $\dot s>0$ is a depreciation; $s\equiv\ln S$
- $Q_{i,t}\equiv P_{i,t}/S_{i,t}$ — CPI-based **real exchange rate**, quoted as USD per local basket, so a rise is a real *appreciation*; $q\equiv\ln Q=p-s$ and $\dot q=\pi-\dot s$. (This is the reciprocal of the textbook $SP^{*}/P$ form; the project is long $Q$.)
- $\tau,\ c,\ E$ — real tariff, real unit cost, constant energy output
- $\rho$ — discount rate; $A_{\rho,n}\equiv\dfrac{1-(1+\rho)^{-n}}{\rho}$ — present value of a unit annuity
- $R_{i,t}\equiv Q_{i,t}/Q_{i,0}-1$ — cumulative (holding-period) return on $Q$ since close
- $D$ — debt stock; $\text{LLCR}$ — loan-life coverage ratio
- $V_i$ — present value of project P&L; $L_i$ — Facility loss; $G$ — the Facility
- $\Psi$ — level annual premium; $\Delta r$ — margin-equivalent interest-rate add-on
- $\mathbb{E}[\cdot]$ — expectation

In the code below, the DataFrame column `Q` holds $Q_{i,t}$ (computed as `cpi / exchange_rate`), and `exchange_rate` holds $S_{i,t}$.


## A stylised project

The analysis below focuses on the CPI-based real exchange rate $Q_{i,t}=P_{i,t}/S_{i,t}$, where $P_i$ is the domestic price level and $S_i$ the nominal exchange rate in LCU per USD. As an index, $Q$ measures the change in the USD value of a fixed local basket of goods. If the local price of electricity is made to grow at the national inflation rate, $Q$ also measures the evolution of the USD value of a MWh. It is easy to see (and we show in detail below) that when tariffs are inflation-adjusted, $Q$ is a good measure of the dollar cash flows available for debt service.

Consider, for simplicity, a project generating a constant amount of energy $E$ from $t=0$ to $T$ at constant real unit cost $c$, excluding hard-currency debt service, and selling at an inflation-indexed real tariff $\tau$. Nominal local-currency revenue is $P_{i,t}\,\tau\,E$ and earnings before debt service are $P_{i,t}(\tau-c)E$. That is, local-currency cash flows available for debt service (CFADS) are a constant growing at the rate of inflation, and CFADS in dollars are

$$
\text{CF}^{\$}_{i,t} \;=\; Q_{i,t}\,(\tau-c)\,E .
$$

For the sake of exposition, we consider a stylised project in which all debt is in hard currency and $\text{CF}^{\$}_{i,0} = Q_{i,0}(\tau-c)E = 100$, where $t=0$ is the first year of debt service: at the close-date exchange rate and price level, 100 USD are available for hard-currency debt service. If $Q$ falls by 10%, only 90 USD are available. Project finance structures usually require a debt service coverage ratio (DSCR) of 120–130%; for ease of exposition we simplify to $\text{DSCR}=1$, assuming away any buffer for FX swings. This implies zero expected profit: any decrease in $Q$ creates a debt service shortfall (and technical default), any increase a surplus. The main affordability result below is only strengthened if $\text{DSCR}>1$.

The surplus (or shortfall, when negative) in USD CFADS in period $t$ is $100\,R_{i,t}$, where

$$
R_{i,t} \;\equiv\; \frac{\text{CF}^{\$}_{i,t}}{\text{CF}^{\$}_{i,0}} - 1 \;=\; \frac{Q_{i,t}}{Q_{i,0}} - 1
$$

is the cumulative return on $Q$ since close — a holding-period return, not a per-period flow. Given a discount rate $\rho$, the present value at close of the lifetime P&L is

$$
V_i \;=\; \int_{0}^{T} \frac{R_{i,t}}{(1+\rho)^{t}}\,dt ,
$$

so the realised profit ($V_i>0$) or loss ($V_i<0$) is $100\,V_i$, the 100 reflecting the constant debt service. Any repayment profile that is more back-loaded (as is common in such transactions) puts more weight on more heavily discounted later periods and thus decreases $|V_i|$.


## The wrong decomposition and the right one

The market treats this exposure as *nominal* FX risk. But nominal depreciation decomposes as

$$
\dot s_{i,t} \;=\; \underbrace{\big(\pi_{i,t} - \pi^{\$}_{t}\big)}_{\text{inflation differential}} \;+\; \underbrace{\big(-\dot q_{i,t}\big)}_{\text{real depreciation}} ,
$$

and an inflation-indexed tariff passes the inflation component straight through to revenues. Because $Q=P/S$, taking logs and differentiating gives the instantaneous identity $\dot q_{i,t} = \pi_{i,t} - \dot s_{i,t}$, the rate of **real appreciation**: inflation net of nominal depreciation. Integrating from close,

$$
\ln\!\big(1+R_{i,t}\big) \;=\; q_{i,t}-q_{i,0} \;=\; \int_{0}^{t}\big(\pi_{i,u}-\dot s_{i,u}\big)\,du .
$$

The per-period P&L is thus the cumulative real appreciation since financial close. The project gains exactly when domestic inflation has cumulatively outpaced nominal depreciation, and falls into shortfall when the currency has depreciated *in real terms*. The inflation level itself drops out — it is passed through by the indexed tariff — leaving only the real component as risk. @sec-hedging shows what it costs to ignore this; the instrument of @sec-instrument prices only the residual.

TODO: tighten prose; add one-sentence roadmap of the paper.


# The real exchange rate: measuring the risk the project actually bears {#sec-index}

Data: WDI CPI and official exchange rate series for African countries, 1985–2024, in `data/wdi_africa_long.csv` (Lesotho excluded for uncorrected CPI basket breaks in 1999, 2002 and 2009). We construct $Q = P/S$ per country and plot it against the nominal rate $S$: the gap between the two lines is the inflation component that indexed revenues reprice automatically.


In [ ]:
#| label: fig-ken
#| fig-cap: "Kenya: nominal exchange rate (USD per KES) against the real exchange rate Q. TODO: pick hlines/shading for the final figure."
country_chart('KEN')

In [ ]:
#| label: fig-nga
#| fig-cap: "Nigeria: nominal exchange rate against Q. The nominal collapse vastly overstates the real one."
country_chart('NGA')

In [ ]:
#| label: tbl-nga-decomp
#| tbl-cap: "Nigeria 2014-2024: decomposition of the nominal move into inflation and the real residual."
# Motivating fact: over 2014-2024 the naira's nominal move was ~9.4x while
# prices moved ~4.8x - roughly half the headline depreciation was inflation
# that an indexed tariff repriced automatically. Q bottoms at ~51, not ~11.
w = df[(df.iso3 == 'NGA') & (df.year.isin([2014, 2024]))].sort_values('year')
s_ratio = w.exchange_rate.iloc[1] / w.exchange_rate.iloc[0]
p_ratio = w.cpi.iloc[1] / w.cpi.iloc[0]
q_ratio = w.Q.iloc[1] / w.Q.iloc[0]
pd.DataFrame({
    'metric': ['Exchange rate (N/USD)', 'Price level (CPI)', 'Q = P/S (real, USD revenue index)'],
    'ratio 2024/2014': [s_ratio, p_ratio, q_ratio],
    'implied annual rate': [s_ratio**0.1 - 1, p_ratio**0.1 - 1, q_ratio**0.1 - 1],
}).round(3)

# Why the standard fixes overcharge: hedging is an inflation bet {#sec-hedging}

Refinancing into local currency and rolling NDF hedges are the same trade: covered interest parity pins the forward rate to the interest differential, $F = S_0(1+i_{LCU})/(1+i_{USD})$, by arbitrage. Both therefore charge the *full* differential

$$ i_{LCU} - i_{USD} \;=\; \underbrace{\mathbb{E}\big[\pi - \pi^{\$}\big]}_{\text{inflation diff.}} \;+\; \underbrace{\mathbb{E}\big[-\dot q\big]}_{\text{real depr.}} \;+\; \underbrace{rp}_{\text{risk premium}} $$

to shed an exposure of which the project — with indexed revenues — only ever bore the real part $-\dot q$.

## The parity accounting

Measured in units of the project's own (CPI-indexed) revenue, the extra cost of the local-currency loan over the USD loan collapses to $rp$ *if expectations are realised*: the inflation component is a wash in present value, paid as a certain, front-loaded coupon against an uncertain, back-loaded erosion of the nominal liability. TODO: appendix derivation (see @sec-appendix).

## The hedger's residual risk position: long local inflation

The refinanced borrower's cash-flow health is $P_{i,t}/P_{i,0}$ against a fixed nominal annuity — only realised inflation pays back the prepaid differential; realised *real* depreciation pays them nothing in cash (it only vindicates the hedge counterfactually). The hedge thus converts FX exposure into a forced bet *on* high local inflation, whose losing branch is a stable, low-inflation currency (the Kenya path, @sec-kenya-refi). The Facility client of @sec-instrument, by contrast, holds no inflation position at all: upside on real appreciation kept, downside on real depreciation insured.


In [ ]:
# TODO (new analysis - anti-hedge exhibit #1): variance decomposition of
# nominal depreciation into inflation differential + real residual.
#
# Spec:
#   1. Add a US CPI series (FRED CPIAUCSL annual, or WDI 'USA' row) -> us_cpi[year].
#   2. For each country and horizon h in {1, 5, 10, 15}:
#        d_log_s  = log S ratio over h years
#        d_log_pi = log (CPI/CPI_US) ratio over h
#        d_log_q  = d_log_s - d_log_pi          (real residual)
#      Share explained by inflation = d_log_pi / d_log_s (report medians,
#      and var decomposition var(s) = var(pi) + var(q) + 2cov across windows).
#   3. Figure: share of cumulative nominal depreciation explained by inflation,
#      by horizon (pooled + per country dots).
# Rebuts the "naira fell 483%" framing (TCX/AfDB 2026) with the fraction
# an indexed tariff already hedges.
pass

## The tilt problem

Even at actuarially fair pricing ($rp = 0$), a 13% nominal coupon is economically "5% real interest + 8% accelerated amortisation": the inflation compensation is paid in cash now, the offsetting erosion of the liability accrues later. Debt service starts far above revenue and only falls below it after several years - so DSCR-constrained leverage drops by roughly a quarter, and equity fills the gap. That cost is paid on *every* path, including the ones where the hedge is fine ex post.

In [ ]:
#| label: fig-tilt
#| fig-cap: "Debt service as a share of (indexed) revenue: flat USD annuity vs local-currency annuity at UIP-fair coupon. The local structure starts ~40% above revenue."
# Stylised tilt chart: USD loan 5%, local coupon 13%, expected inflation ~9%.
# Both loans have the same PV; the local annuity front-loads real debt service.
# USD loan: flat payment vs revenue flat in real USD terms -> ratio 1 throughout.
# LCU loan: fixed nominal annuity vs revenue growing with CPI -> starts ~40% above.
T_l, r_usd, r_lcu, infl = 10, 0.05, 0.13, 0.09
af_usd = (1 - (1 + r_usd) ** -T_l) / r_usd
af_lcu = (1 - (1 + r_lcu) ** -T_l) / r_lcu
years = np.arange(1, T_l + 1)
rev = (1 + infl) ** years            # indexed revenue in LCU, rev_0 = 1
pay_lcu = af_usd / af_lcu            # fixed LCU annuity per unit of USD-loan payment, at close FX
tilt = pd.DataFrame({
    'year': np.tile(years, 2),
    'ds_over_revenue': np.concatenate([np.ones(T_l), pay_lcu / rev]),
    'structure': ['USD loan (flat 100/yr)'] * T_l + ['LCU loan (UIP-fair coupon)'] * T_l,
})
base = alt.Chart(tilt).mark_line(point=True).encode(
    x=alt.X('year:O', title='Year of operation'),
    y=alt.Y('ds_over_revenue:Q', title='Debt service / revenue', axis=alt.Axis(format='%')),
    color=alt.Color('structure:N', legend=alt.Legend(orient='bottom', title=None)),
)
rule = alt.Chart(pd.DataFrame({'y': [1.0]})).mark_rule(color='#888', strokeDash=[4, 4]).encode(y='y:Q')
(base + rule).properties(width=550, height=280)

# The instrument: an emergency liquidity facility for the real residual {#sec-instrument}

Let the **Facility** $G$ provide emergency liquidity. Whenever $R_{i,t}<0$, $G$ advances $100\,|R_{i,t}|$ so the project can meet its 100 of debt service; whenever $R_{i,t}>0$, the project escrows the surplus in an interest-bearing USD account to repay $G$ at $T$. Equivalently, $G$ tops the project's coverage back up to $\text{DSCR}=1$ in every period. Netting advances against escrowed surpluses, $G$'s present-value payout is

$$
L_i \;=\; 100\,(-V_i)^{+}, \qquad \mathbb{E}[L_i] \;=\; -\,100\,\mathbb{E}[V_i]\ \text{ (mean basis)} ,
$$

so $G$ is made whole with interest when $V_i\ge 0$ and absorbs $100\,|V_i|$ when $V_i<0$. The actuarially fair premium covers $\mathbb{E}[L_i]$, abstracting from solvency capital (@sec-solvency) and profit loadings.

**Cost relative to the debt stock, and the LLCR.** Since debt service is $n$ payments of 100, the amount borrowed is their present value,

$$
D \;=\; 100\,A_{\rho,n}, \qquad A_{\rho,n}=\frac{1-(1+\rho)^{-n}}{\rho} .
$$

The debt stock is the natural denominator, because the cover is in effect insurance on the loan-life coverage ratio. The present value of dollar CFADS over the loan life is $\int_0^T 100(1+R_{i,t})(1+\rho)^{-t}\,dt = D + 100\,V_i$, so

$$
\text{LLCR}_i \;=\; \frac{\text{PV}_0\!\big(\text{CFADS}^{\$}\big)}{D} \;=\; 1 + \frac{100\,V_i}{D} \;=\; 1 + \frac{V_i}{A_{\rho,n}} .
$$

The instrument draws exactly when $\text{LLCR}_i<1$ and tops it back to one. Its expected cost per dollar of debt is therefore the expected shortfall of the loan-life coverage ratio below unity:

$$
\frac{\mathbb{E}[L_i]}{D} \;=\; -\,\frac{\mathbb{E}[V_i]}{A_{\rho,n}} \;=\; \mathbb{E}\big[\,1-\text{LLCR}_i\,\big]
$$

(with the option-aware refinement $\mathbb{E}\big[(1-\text{LLCR}_i)^{+}\big]$ once the asymmetry of $L_i$ is priced).

**The three premium quotes (and the $A$-identity).** Multiplying by $A_{\rho,n}$ converts an annuity into a present value; dividing converts a present value into an annuity. The actuarially fair *level annual premium* is therefore

$$
\Psi \;=\; \frac{\mathbb{E}[L_i]}{A_{\rho,n}} \;=\; -\,\frac{100\,\mathbb{E}[V_i]}{A_{\rho,n}},
\qquad
\frac{\Psi}{100} \;=\; -\,\frac{\mathbb{E}[V_i]}{A_{\rho,n}} .
$$

So $-\mathbb{E}[V_i]/A_{\rho,n}$ is simultaneously the premium as a fraction of each **annual debt service payment** and — because $D = 100\,A_{\rho,n}$ — the expected PV loss as a fraction of the **debt stock**. One number, two readings: this is the $A$-identity, and it is the pivot between the actuarial and the credit quote.

Dividing by $A_{\rho,n}$ a second time gives

$$
\frac{\Psi}{D} \;=\; -\,\frac{\mathbb{E}[V_i]}{A_{\rho,n}^{2}} ,
$$

a flat annual payment expressed as a share of the *original principal*. **This is not the equivalent interest-rate spread, except on a bullet loan.** On an annuity-amortising loan a margin accrues on the declining outstanding balance — on average roughly 65% of the original principal — so the margin-equivalent add-on $\Delta r$ is about half again as large. It is computed exactly by `equivalent_rate()` in @sec-method, which treats the expected loss as additional principal and solves

$$
A_{\rho+\Delta r,\;n} \;=\; \frac{A_{\rho,n}}{1 + \big|\,\mathbb{E}[V_i]/A_{\rho,n}\,\big|} .
$$

We quote the premium to actuaries as a share of the covered exposure (debt service), and to investors as the margin-equivalent $\Delta r$.


## Risk positions compared

TODO (Fig 4.1, schematic - probably a static drawing rather than code): three-row
payoff diagram - naked USD / hedged / USD + facility - against the two shocks
(inflation, real depreciation), showing the facility as the only structure
orthogonal to inflation: the hedger must root for inflation; the facility client
keeps real-appreciation upside and has real-depreciation downside truncated.

# Data and method {#sec-method}

For every country and every rolling 15-year window since 1995, we compute the discounted sum of cumulative returns $R_{i,t} = Q_{i,t}/Q_{i,0} - 1$ — the present value of the lifetime P&L, $V_i$ — and normalise by the annuity factor $A_{\rho,n}$ into a PV per unit of debt stock, i.e. the expected LLCR shortfall of @sec-instrument. `compute_terminal_residuals_wdi` below mirrors `source/data_analysis_functions.py::compute_terminal_residuals`, operating directly on the WDI data; `equivalent_rate` performs the amortising-loan margin conversion to $\Delta r$.


In [ ]:
#| include: false
# Ported from wdi_hybrid_analysis.ipynb, cell 3.
def compute_terminal_residuals_wdi(
    df,
    years=15,
    discount_rate=0.05,
    window_mode="rolling", # or "two_non_overlapping",  # 
    col="Q",
    start_year=None,
):
    """Mirror of source.data_analysis_functions.compute_terminal_residuals,
    operating directly on the wdi_africa_long-shaped df with columns:
    iso3, country, year, exchange_rate, cpi, usd_per_lcu, Q.

    For each country, slides a fixed-horizon window over the series in `col`,
    computes R_it = v/v[t0] - 1, discounts year-end cashflows back to start,
    and returns the present value of lifetime P&L, V_i, normalised by the
    annuity factor A to a PV per unit of debt stock.
    If `start_year` is given, df is filtered to year >= start_year before
    any windowing.
    """
    if start_year is not None:
        df = df[df["year"] >= start_year]

    af = (1 - (1 + discount_rate) ** -years) / discount_rate
    out = []

    for iso3, g in df.dropna(subset=[col]).groupby("iso3"):
        g = g.sort_values("year").reset_index(drop=True)
        n = len(g)
        if n < years:
            continue

        if window_mode == "two_non_overlapping":
            starts = list(range(0, n - years + 1, years))[:2]
        elif window_mode == "rolling":
            starts = list(range(0, n - years + 1))
        else:
            raise ValueError("window_mode must be 'two_non_overlapping' or 'rolling'")

        for i0 in starts:
            win = g.iloc[i0:i0 + years]
            if win[col].isna().any():
                continue

            base = float(win[col].iloc[0])
            R_it = (win[col].to_numpy() / base) - 1.0
            t = np.arange(1, years + 1, dtype=float)
            dfactors = (1.0 + discount_rate) ** (-t)

            V_i = float(np.sum(R_it * dfactors))
            net_pv_per_debt_stock = V_i / af
            loss = min(net_pv_per_debt_stock, 0.0)

            out.append({
                "iso3": iso3,
                "country": win["country"].iloc[0],
                "window_start": int(win["year"].iloc[0]),
                "window_end": int(win["year"].iloc[-1]),
                "baseline": base,
                "net_pv_by_debt_stock": net_pv_per_debt_stock,
                "loss": loss,
            })

    return pd.DataFrame(out)


from scipy.optimize import brentq


def equivalent_rate(expected_loss, discount_rate, years=15):
    """Translate an expected PV loss (fraction of debt stock) into the
    equivalent interest rate.

    The expected loss is treated as additional debt, so the inflated principal
    is (1 + |expected_loss|). Solve for the rate r_new at which the annuity on
    the original principal (=1) equals the annuity on the inflated principal
    priced at the original rate, i.e.
        af(r_new, years) == af0 / (1 + |loss|),
    where `af0` is the original annuity factor. af is strictly decreasing in r,
    so the root is unique.
    """
    af0 = (1 - (1 + discount_rate) ** -years) / discount_rate
    target_af = af0 / (1.0 + abs(expected_loss))
    af = lambda r: (1.0 - (1.0 + r) ** -years) / r
    return brentq(lambda r: af(r) - target_af, 1e-9, 1.0)

In [ ]:
#| label: fig-windows-nga
#| fig-cap: "Nigeria: Q (indexed to first year) with every rolling 15-year window's start level and its terminal residual, green = gain, red = loss, shaded by window start."
# Ported from wdi_hybrid_analysis.ipynb, cell 6 (styled windows chart).
# --- Redesigned colored-windows chart (new version, for comparison with the cell above) ---
# Color = green (gain) / red (loss) with a light->dark shade by window start within each hue,
# used identically on each window's horizontal start-level marker, its end dot on the Q line,
# and its residual bar below. No gridlines; a yearly x-axis (rotated -60) labelled on the
# bottom panel only; the dashed-100 (top) and zero (bottom) reference lines are the anchors.

def _ramp(t, light, dark):
    """Hex color interpolated between `light` (t=0) and `dark` (t=1) in RGB."""
    lo = tuple(int(light[i:i + 2], 16) for i in (1, 3, 5))
    hi = tuple(int(dark[i:i + 2], 16) for i in (1, 3, 5))
    return '#%02x%02x%02x' % tuple(round(a + (b - a) * t) for a, b in zip(lo, hi))


def country_windows_styled(iso3, years=window_length, discount_rate=discount_rate,
                           step=1, first_start=start_year, ypad_frac=0.05):
    res = compute_terminal_residuals_wdi(
        df[df['iso3'] == iso3], years=years, discount_rate=discount_rate,
        window_mode="rolling", col="Q", start_year=first_start).reset_index(drop=True)
    if step > 1:  # thin the rolling windows out if requested
        res = res[(res['window_start'] - res['window_start'].min()) % step == 0].reset_index(drop=True)

    g = (df[(df['iso3'] == iso3) & (df['year'] >= first_start)]
         .dropna(subset=['Q']).sort_values('year'))
    country_name = g['country'].iloc[0]
    q0 = float(g['Q'].iloc[0])
    base_year, last_year = int(g['year'].iloc[0]), int(g['year'].max())

    g = g.assign(idx=g['Q'] / q0 * 100.0)            # index = 100 in the first year
    idx_by_year = dict(zip(g['year'], g['idx']))

    # bivariate color: hue = sign of residual (green gain / red loss); within each hue,
    # lightness ranks the window by start date (light = earliest of that colour -> dark =
    # latest). Normalising per colour (not over all windows) keeps the gradient visible even
    # when one colour has only a few, tightly-clustered windows (e.g. a handful of late losses).
    s = res['window_start']
    sign_pos = res['net_pv_by_debt_stock'] >= 0

    def _shade(mask):
        sub = s[mask]
        if len(sub) == 0:
            return {}
        lo, sp = sub.min(), max(int(sub.max() - sub.min()), 1)
        return {int(ws): 0.25 + 0.6 * (ws - lo) / sp for ws in sub}

    gt, lt = _shade(sign_pos), _shade(~sign_pos)
    fills = [
        _ramp(gt[int(ws)], '#bae4b3', '#005a32') if resid >= 0
        else _ramp(lt[int(ws)], '#fcae91', '#a50f15')
        for ws, resid in zip(s, res['net_pv_by_debt_stock'])
    ]

    wins = res.assign(
        residual=res['net_pv_by_debt_stock'],
        start_idx=res['baseline'] / q0 * 100.0,
        end_idx=res['window_end'].map(idx_by_year),
        fill=fills)

    xscale = alt.Scale(domain=[base_year, last_year + 1])
    ypad = ypad_frac * (g['idx'].max() - g['idx'].min())
    yscale = alt.Scale(domain=[g['idx'].min() - ypad, g['idx'].max() + ypad],
                       zero=False, nice=False)
    year_vals = list(range(base_year, last_year + 1))
    fill_enc = alt.Color('fill:N', scale=None)
    tip = ['window_start', 'window_end', alt.Tooltip('residual:Q', format='+.4f')]

    # Layers share a scale per channel, so every layer must carry the SAME axis spec:
    # axis=None on any layer drops the guide, and differing fields with default titles make
    # Vega blank the title -- so define each axis once and reuse it everywhere.
    top_x = alt.Axis(values=year_vals, labels=False, ticks=True, title=None)  # ticks only
    top_y = alt.Axis(title=f'Q index (=100 in {base_year})')
    bot_x = alt.Axis(values=year_vals, labelAngle=-60, format='d', title='year window ends')
    bot_y = alt.Axis(title='terminal residual / debt stock')

    # --- top panel: Q index, window start-level markers + end dots (full-opacity so their
    #     gradient matches the residual bars below) ---
    ref100 = alt.Chart(pd.DataFrame({'y': [100]})).mark_rule(
        color='#bbb', strokeDash=[4, 4]).encode(y=alt.Y('y:Q', scale=yscale, axis=top_y))
    segs = alt.Chart(wins).mark_rule(strokeWidth=3).encode(
        x=alt.X('window_start:Q', scale=xscale, axis=top_x), x2='window_end:Q',
        y=alt.Y('start_idx:Q', scale=yscale, axis=top_y), color=fill_enc, tooltip=tip)
    hline = alt.Chart(g).mark_line(color='#222', strokeWidth=2).encode(
        x=alt.X('year:Q', scale=xscale, axis=top_x),
        y=alt.Y('idx:Q', scale=yscale, axis=top_y),
        tooltip=['year', alt.Tooltip('idx:Q', format='.1f')])
    dots = alt.Chart(wins).mark_point(filled=True, size=45).encode(
        x=alt.X('window_end:Q', scale=xscale, axis=top_x),
        y=alt.Y('end_idx:Q', scale=yscale, axis=top_y), color=fill_enc, tooltip=tip)
    top = (ref100 + segs + hline + dots).properties(
        width=720, height=300,
        title=alt.TitleParams(
            f"{country_name}: real exchange rate Q (=100 in {base_year})",
            subtitle="horizontal bar = each 15-yr window's start level, dot = where Q ends; "
                     "green = gain, red = loss (darker = later window)",
            anchor='start'))

    # --- bottom panel: terminal residual per window, at window end ---
    zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='#888').encode(
        y=alt.Y('y:Q', axis=bot_y))
    bars = alt.Chart(wins).mark_bar(width=10).encode(
        x=alt.X('window_end:Q', scale=xscale, axis=bot_x),
        y=alt.Y('residual:Q', axis=bot_y), color=fill_enc, tooltip=tip)
    bottom = (zero + bars).properties(width=720, height=160)

    return (alt.vconcat(top, bottom).resolve_scale(x='shared')
            .configure_view(strokeWidth=0)
            .configure_axis(grid=False, domainColor='#888', tickColor='#888'))


country_windows_styled('NGA')


In [ ]:
# TODO (new analysis): mean reversion of Q.
#
# Spec: estimate AR(1) on log Q by country (and pooled panel with country FE);
# report implied half-lives. This underpins why the real residual is insurable
# (long-run relative PPP) while the nominal rate is not, and supports
# out-of-sample validity of the premium. One table: country, rho_hat, half-life.
pass

# Results: the cover costs basis points {#sec-results}

Pooled over all rolling 15-year windows (~718 windows, 50 countries, $\rho = 5\%$): expected loss $\approx$ 1.95% of debt stock in PV, with only ~18% of windows producing any loss. Quoted three ways: **~1.9% of each debt-service payment** (the actuarial quote), **~19bp of principal per year flat**, or **~29bp on the loan margin** of a 15-year annuity loan (19bp for a bullet).

In [ ]:
#| label: fig-hist-pooled
#| fig-cap: "Distribution of terminal residuals across all rolling 15-year windows; the red mass left of zero is the insurer's loss tail."
# Ported from wdi_hybrid_analysis.ipynb, cell 7.
residuals = compute_terminal_residuals_wdi(df, years=window_length, discount_rate=discount_rate, window_mode="rolling", col="Q", start_year=start_year)
af = (1 - (1 + discount_rate) ** -window_length) / discount_rate
expected_loss = residuals["loss"].mean()
prob_loss = (residuals["net_pv_by_debt_stock"] < 0).mean()
n_windows = len(residuals)
n_countries = residuals["iso3"].nunique()
r_new = equivalent_rate(expected_loss=expected_loss, discount_rate=discount_rate, years=window_length)


print(f"Windows from {start_year} to 2024")
print(f"Number of windows  : {n_windows}")
print(f"Number of countries: {n_countries}")
print(f"Expected loss      : {expected_loss:.4f}")
print(f"Probability of loss: {prob_loss:.4f}")
print(f"interest rate increase: {r_new-discount_rate}")
print(f"annual premium: {expected_loss/af} times debt stock")

hist = alt.Chart(residuals).mark_bar().encode(
    x=alt.X('net_pv_by_debt_stock:Q', bin=alt.Bin(maxbins=40), title='V / debt stock  (PV of lifetime P&L)'),
    y=alt.Y('count():Q', title=f'Number of {window_length} year windows in pool'),
    color=alt.condition(
        alt.datum.net_pv_by_debt_stock < 0,
        alt.value('crimson'),
        alt.value('steelblue'),
    ),
).properties(
    width=600,
    height=350,
    title=f'Distribution of terminal residuals (n={n_windows}, expected loss={expected_loss:.3f}, P(loss)={prob_loss:.2%})'
)
hist

In [ ]:
#| label: fig-premium-by-country
#| fig-cap: "Expected loss by country, as a fraction of the annuity (left axis) and of the debt stock (right axis)."
# Ported from wdi_hybrid_analysis.ipynb, cell 8.
by_country = (
    residuals.groupby(["iso3", "country"], as_index=False)
    .agg(
        expected_loss=("loss", "mean"),
        prob_loss=("net_pv_by_debt_stock", lambda s: (s < 0).mean()),
        n_windows=("loss", "size"),
    )
    .sort_values("expected_loss")  # most negative first = highest expected loss
    .reset_index(drop=True)
)
# Equivalent rate increase: the additional interest rate (over the original
# discount_rate) that would impose the same PV burden as the expected loss.
by_country["rate_increase"] = by_country["expected_loss"].apply(
    lambda el: equivalent_rate(el, discount_rate=discount_rate, years=window_length) - discount_rate
)
# expected_loss is net PV normalized by the annuity factor (af), i.e. a fraction
# of the annuity. Dividing by af again expresses it as a fraction of debt stock.
by_country["expected_loss_debt"] = by_country["expected_loss"] / af
print(by_country.to_string(index=False))

sort_order = by_country["country"].tolist()
shared_tooltip = ["country", alt.Tooltip("expected_loss:Q", format=".4f"),
                  alt.Tooltip("expected_loss_debt:Q", format=".4f"),
                  alt.Tooltip("prob_loss:Q", format=".2%"),
                  alt.Tooltip("rate_increase:Q", format=".4f"),
                  "n_windows"]

# Left axis: fraction of the annuity (the bars actually drawn).
bar_annuity = alt.Chart(by_country).mark_bar(color="crimson").encode(
    x=alt.X("country:N", sort=sort_order, title="Country", axis=alt.Axis(labelAngle=-60)),
    y=alt.Y("expected_loss:Q", title="Expected loss (fraction of annuity)"),
    tooltip=shared_tooltip,
)

# Right axis: same loss expressed as a fraction of the debt stock (= /af).
# Drawn invisibly so it only contributes the secondary axis.
bar_debt = alt.Chart(by_country).mark_bar(opacity=0).encode(
    x=alt.X("country:N", sort=sort_order),
    y=alt.Y("expected_loss_debt:Q",
            title="Expected loss (fraction of debt stock)",
            axis=alt.Axis(orient="right")),
    tooltip=shared_tooltip,
)

bar = alt.layer(bar_annuity, bar_debt).resolve_scale(y="independent").properties(
    width=800,
    height=400,
    title="Expected loss by country (sorted: highest expected loss → lowest)",
)
bar

In [ ]:
#| label: fig-rate-by-country
#| fig-cap: "Equivalent margin uplift by country (annuity-amortising loan) - the investor-facing number."
# Ported from wdi_hybrid_analysis.ipynb, cell 9.
rate_sorted = by_country.sort_values("rate_increase", ascending=False).reset_index(drop=True)

rate_bar = alt.Chart(rate_sorted).mark_bar().encode(
    x=alt.X("country:N", sort=rate_sorted["country"].tolist(), title="Country", axis=alt.Axis(labelAngle=-60)),
    y=alt.Y("rate_increase:Q", title="Equivalent rate increase (over original interest rate)",
            axis=alt.Axis(format="%")),
    tooltip=["country", alt.Tooltip("rate_increase:Q", format=".4f"),
             alt.Tooltip("expected_loss:Q", format=".4f"),
             alt.Tooltip("prob_loss:Q", format=".2%"),
             "n_windows"],
    color=alt.value("darkorange"),
).properties(
    width=800,
    height=400,
    title="Equivalent rate increase by country (sorted: highest → lowest)",
)
rate_bar

In [ ]:
# TODO (new analysis - robustness): sample-start sensitivity, including the
# 1994 CFA devaluation.
#
# Spec: rerun the pooled premium with start_year in {1985, 1990, 1995, 2000}
# (data reaches back to 1985 for most countries). One table: start year,
# n windows, expected loss, P(loss), margin-equivalent. The 1995 start sits
# one year after the 50% CFA devaluation - this table preempts that critique.
pass

# Country variation, regimes, and diversification {#sec-variation}

Expected loss ranges from exactly zero (Kenya: no window since 1995 ends PV-net below its start) to ~97bp of principal per year (Nigeria, where every losing window straddles the 1999 unification). Countries are classified into three exchange-rate regime groups (IMF AREAER, predominant arrangement 2000-2023): EUR-linked (CFA zone + Comoros, Cape Verde), ZAR-linked (CMA), and Flexible.

In [ ]:
#| label: tbl-country-stats
#| tbl-cap: "Country-level premium statistics by regime."
# Adapted from consolidated_currency_analysis.ipynb, cell 7
# (regime mapped on iso3; note: overlaps with by_country above - merge later).
residuals['regime'] = residuals['iso3'].map(fx_regime).fillna('Flexible')

country_stats = residuals.groupby(['iso3', 'country']).agg(
    n_windows=('loss', 'count'),
    premium=('loss', 'mean'),
    prob_loss=('loss', lambda x: (x < 0).mean()),
    e_loss_given_loss=('loss', lambda x: x[x < 0].mean() if (x < 0).any() else 0.0),
    worst=('loss', 'min'),
    best=('net_pv_by_debt_stock', 'max'),
).reset_index()
country_stats['regime'] = country_stats['iso3'].map(fx_regime).fillna('Flexible')
country_stats = country_stats.sort_values('premium')

pooled_premium = residuals['loss'].mean()

display_df = country_stats[['country', 'regime', 'n_windows', 'premium', 'prob_loss', 'e_loss_given_loss', 'worst', 'best']].copy()
display_df.columns = ['Country', 'Regime', 'Windows', 'Premium (%)', 'P(loss)', 'E[loss|loss] (%)', 'Worst (%)', 'Best (%)']
for col in ['Premium (%)', 'E[loss|loss] (%)', 'Worst (%)', 'Best (%)']:
    display_df[col] = display_df[col] * 100
display_df['P(loss)'] = display_df['P(loss)'] * 100

print(f"Pooled annual premium: {pooled_premium*100:.3f}% of debt stock")
print(f"Countries: {len(country_stats)}, Total windows: {residuals.shape[0]}\n")
print(display_df.to_string(index=False, float_format='%.3f'))

In [ ]:
#| label: tbl-regime-summary
#| tbl-cap: "Regime-level summary."
# Ported from consolidated_currency_analysis.ipynb, cell 11.
# Regime-level summary table
regime_rows = []
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible', 'All']:
    r = residuals if regime == 'All' else residuals[residuals['regime'] == regime]
    countries_list = sorted(r['country'].unique())
    regime_rows.append({
        'Regime': regime,
        'Countries': r['country'].nunique(),
        'Windows': len(r),
        'Premium (%)': r['loss'].mean() * 100,
        'P(loss) (%)': (r['loss'] < 0).mean() * 100,
        'E[loss|loss] (%)': r.loc[r['loss'] < 0, 'loss'].mean() * 100 if (r['loss'] < 0).any() else 0,
        'Worst (%)': r['loss'].min() * 100,
    })

regime_summary = pd.DataFrame(regime_rows)
print("Regime-level summary")
print("=" * 80)
print(regime_summary.to_string(index=False, float_format='%.3f'))

# Also list countries per regime
print("\nCountries by regime:")
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible']:
    countries_in = sorted(residuals[residuals['regime'] == regime]['country'].unique())
    print(f"  {regime} ({len(countries_in)}): {', '.join(countries_in)}")

In [ ]:
#| label: fig-premium-by-regime
#| fig-cap: "Country premium distribution by regime (red dashed = pooled)."
regime_order = ['EUR-linked', 'ZAR-linked', 'Flexible']
regime_colors = ['#4c78a8', '#f58518', '#54a24b']
# Ported from consolidated_currency_analysis.ipynb, cell 13.
# Box + strip plot: country-level premiums by regime
box_data = country_stats[['country', 'regime', 'premium']].copy()
box_data['premium_pct'] = box_data['premium'] * 100

boxes = alt.Chart(box_data).mark_boxplot(extent='min-max', size=40).encode(
    x=alt.X('regime:N', sort=regime_order, title='Regime'),
    y=alt.Y('premium_pct:Q', title='Annual premium (% of debt stock)'),
    color=alt.Color('regime:N', scale=alt.Scale(domain=regime_order, range=regime_colors), legend=None)
)

points = alt.Chart(box_data).mark_circle(size=50, opacity=0.6).encode(
    x=alt.X('regime:N', sort=regime_order),
    y=alt.Y('premium_pct:Q'),
    color=alt.Color('regime:N', scale=alt.Scale(domain=regime_order, range=regime_colors), legend=None),
    tooltip=['country', alt.Tooltip('premium_pct:Q', format='.3f', title='Premium (%)')]
)

pooled_line = alt.Chart(pd.DataFrame({'y': [pooled_premium * 100]})).mark_rule(
    color='red', strokeDash=[4, 4], strokeWidth=1.5
).encode(y='y:Q')

(boxes + points + pooled_line).properties(
    width=300, height=300,
    title='Country Premium Distribution by Regime (red dashed = pooled)'
)

In [ ]:
#| label: fig-corr-heatmap
#| fig-cap: "Pairwise correlation of annual real exchange rate changes, ordered by regime (black lines = regime boundaries)."
# Adapted from consolidated_currency_analysis.ipynb, cell 17 (WDI columns).
# Correlation heatmap of annual Q changes, ordered by regime
wide = df.pivot(index='year', columns='iso3', values='Q')
# NB: .dropna() here would drop every row (country coverage differs), leaving an
# all-NaN matrix; use pairwise-complete correlations instead.
returns = wide.pct_change(fill_method=None)
corr = returns.corr(min_periods=10)

# Order countries by regime
eur_countries = sorted([c for c in corr.columns if fx_regime.get(c) == 'EUR-linked'])
zar_countries = sorted([c for c in corr.columns if fx_regime.get(c) == 'ZAR-linked'])
flex_countries = sorted([c for c in corr.columns if c not in eur_countries + zar_countries])
ordered = eur_countries + zar_countries + flex_countries

# Melt to long format
corr_ordered = corr.loc[ordered, ordered]
corr_long = corr_ordered.reset_index().melt(id_vars='iso3', var_name='country2', value_name='correlation')
corr_long = corr_long.rename(columns={'iso3': 'country1'})

heatmap = alt.Chart(corr_long).mark_rect().encode(
    x=alt.X('country2:N', sort=ordered, title=None),
    y=alt.Y('country1:N', sort=ordered, title=None),
    color=alt.Color('correlation:Q',
                    scale=alt.Scale(scheme='redblue', domain=[-0.5, 1], domainMid=0),
                    title='Corr'),
    tooltip=['country1', 'country2', alt.Tooltip('correlation:Q', format='.2f')]
).properties(width=500, height=500, title='Pairwise Correlation of Annual Real Exchange Rate Changes')

# Regime boundary lines
n_eur = len(eur_countries)
n_zar = len(zar_countries)

boundary_data = pd.DataFrame({'pos': [n_eur - 0.5, n_eur + n_zar - 0.5]})
vrules = alt.Chart(boundary_data).mark_rule(color='black', strokeWidth=1.5).encode(x='pos:Q')
hrules = alt.Chart(boundary_data).mark_rule(color='black', strokeWidth=1.5).encode(y='pos:Q')

(heatmap + vrules + hrules).configure_axis(labelFontSize=6)

In [ ]:
#| label: tbl-mean-corr
#| tbl-cap: "Mean pairwise correlations within and across regime groups."
# Ported from consolidated_currency_analysis.ipynb, cell 18.
# Mean pairwise correlation statistics
def mean_pairwise_corr(corr_matrix, group):
    sub = corr_matrix.loc[group, group]
    mask = np.triu(np.ones(sub.shape, dtype=bool), k=1)
    vals = sub.where(mask).stack()
    return vals.mean(), vals.std(), len(vals)

def mean_cross_corr(corr_matrix, g1, g2):
    vals = corr_matrix.loc[g1, g2].stack()
    return vals.mean(), vals.std(), len(vals)

print("Mean pairwise correlation of annual Q changes")
print("=" * 60)

corr_rows = []
m, s, n = mean_pairwise_corr(corr, eur_countries)
corr_rows.append({'Group': 'Within EUR-linked', 'Mean corr': m, 'Std': s, 'Pairs': n})

if len(zar_countries) > 1:
    m, s, n = mean_pairwise_corr(corr, zar_countries)
    corr_rows.append({'Group': 'Within ZAR-linked', 'Mean corr': m, 'Std': s, 'Pairs': n})

m, s, n = mean_pairwise_corr(corr, flex_countries)
corr_rows.append({'Group': 'Within Flexible', 'Mean corr': m, 'Std': s, 'Pairs': n})

m, s, n = mean_cross_corr(corr, eur_countries, flex_countries)
corr_rows.append({'Group': 'EUR-linked vs Flexible', 'Mean corr': m, 'Std': s, 'Pairs': n})

if len(zar_countries) > 0:
    m, s, n = mean_cross_corr(corr, zar_countries, flex_countries)
    corr_rows.append({'Group': 'ZAR-linked vs Flexible', 'Mean corr': m, 'Std': s, 'Pairs': n})
    m, s, n = mean_cross_corr(corr, eur_countries, zar_countries)
    corr_rows.append({'Group': 'EUR-linked vs ZAR-linked', 'Mean corr': m, 'Std': s, 'Pairs': n})

corr_table = pd.DataFrame(corr_rows)
print(corr_table.to_string(index=False, float_format='%.3f'))

In [ ]:
# Diversification benefit: pooled vs average country-level.
# Ported from consolidated_currency_analysis.ipynb, cell 19.
# Diversification benefit: pooled vs average country-level
avg_country_premium = country_stats['premium'].mean()
avg_country_ploss = country_stats['prob_loss'].mean()

pooled_ploss = (residuals['loss'] < 0).mean()

print("Diversification benefit")
print("=" * 60)
print(f"{'Metric':<35} {'Pooled':>12} {'Avg country':>12}")
print("-" * 60)
print(f"{'Annual premium (% debt stock)':<35} {pooled_premium*100:>12.3f} {avg_country_premium*100:>12.3f}")
print(f"{'P(loss)':<35} {pooled_ploss*100:>12.1f}% {avg_country_ploss*100:>12.1f}%")
print()
print("The pooled premium equals the average country premium (the expectation")
print("is additive). The diversification benefit comes from variance reduction:")
print("a facility covering many countries faces a tighter loss distribution,")
print("reducing capital requirements.")

In [ ]:
#| label: fig-bootstrap
#| fig-cap: "Bootstrap distributions of the pooled premium (country-block resampling, 10,000 draws)."
# Ported from consolidated_currency_analysis.ipynb, cell 20.
# Bootstrap confidence intervals (country-block resampling, 10,000 draws)
def bootstrap_premium(res_df, n_boot=10000, seed=42):
    rng = np.random.default_rng(seed)
    countries = res_df['country'].unique()
    n = len(countries)
    country_losses = {c: res_df.loc[res_df['country'] == c, 'loss'].values for c in countries}
    boot = np.empty(n_boot)
    for b in range(n_boot):
        sampled = rng.choice(countries, size=n, replace=True)
        all_losses = np.concatenate([country_losses[c] for c in sampled])
        boot[b] = all_losses.mean()
    point = res_df['loss'].mean()
    return {'point': point, 'se': boot.std(),
            'ci_lo': np.percentile(boot, 2.5), 'ci_hi': np.percentile(boot, 97.5),
            'distribution': boot}

# Run bootstrap for overall and per-regime
boot_results = {}
boot_results['All'] = bootstrap_premium(residuals)
for regime in ['EUR-linked', 'ZAR-linked', 'Flexible']:
    boot_results[regime] = bootstrap_premium(residuals[residuals['regime'] == regime])

# Print summary
print("Bootstrap confidence intervals (10,000 country-block resamples)")
print("=" * 70)
for group, b in boot_results.items():
    print(f"{group:>12}: premium = {b['point']*100:.3f}%  "
          f"SE = {b['se']*100:.3f}%  "
          f"95% CI = [{b['ci_lo']*100:.3f}%, {b['ci_hi']*100:.3f}%]")

# Pre-bin bootstrap distributions in pandas to avoid Altair's 5000-row limit
group_order = ['All', 'EUR-linked', 'ZAR-linked', 'Flexible']
group_colors = ['steelblue', '#4c78a8', '#f58518', '#54a24b']

binned_rows = []
annot_rows = []
for group, b in boot_results.items():
    dist = b['distribution'] * 100
    counts, edges = np.histogram(dist, bins=50)
    centers = (edges[:-1] + edges[1:]) / 2
    for c, cnt in zip(centers, counts):
        if cnt > 0:
            binned_rows.append({'group': group, 'bin_center': c, 'count': int(cnt)})
    annot_rows.append({'group': group, 'x': b['point'] * 100, 'type': 'point'})
    annot_rows.append({'group': group, 'x': b['ci_lo'] * 100, 'type': 'ci'})
    annot_rows.append({'group': group, 'x': b['ci_hi'] * 100, 'type': 'ci'})

binned_df = pd.DataFrame(binned_rows)
annot_df = pd.DataFrame(annot_rows)

# Merge annotations into binned data for faceted layering
binned_df = binned_df.merge(
    annot_df[annot_df['type'] == 'point'][['group', 'x']].rename(columns={'x': 'point_est'}),
    on='group'
)
binned_df = binned_df.merge(
    annot_df[annot_df['type'] == 'ci'].groupby('group')['x'].agg(['min', 'max']).rename(
        columns={'min': 'ci_lo', 'max': 'ci_hi'}),
    on='group'
)

base = alt.Chart(binned_df)

hist_chart = base.mark_bar(opacity=0.8).encode(
    x=alt.X('bin_center:Q', title='Annual premium (% of debt stock)'),
    y=alt.Y('count:Q', title='Frequency'),
    color=alt.Color('group:N', scale=alt.Scale(domain=group_order, range=group_colors), legend=None)
)

point_rules = base.mark_rule(color='red', strokeDash=[4, 4], strokeWidth=1.5).encode(
    x='point_est:Q'
)

ci_lo_rules = base.mark_rule(color='gray', strokeDash=[2, 2], strokeWidth=1).encode(
    x='ci_lo:Q'
)

ci_hi_rules = base.mark_rule(color='gray', strokeDash=[2, 2], strokeWidth=1).encode(
    x='ci_hi:Q'
)

(hist_chart + point_rules + ci_lo_rules + ci_hi_rules).facet(
    facet=alt.Facet('group:N', title='Group', sort=group_order),
    columns=2
).properties(title='Bootstrap Distributions of Annual FX Risk Premium')

# Worked examples: how the cover pans out for an investor {#sec-examples}

At financial close, sponsors commonly plan to refinance their hard-currency exposure quickly. The examples below run the actual data through three structures - naked USD debt, USD debt + facility, and a local-currency refinance at close - for the paths investors fear (Nigeria 2014) and the paths where the hedge is pure cost (Kenya 2009). All ledgers are in USD, discounted at the USD rate (5%), against the baseline of the original USD loan (principal 772, flat debt service 100/yr over 10 years).

## Nigeria 2014: the path the refinance-minded investor fears {#sec-nga-ledger}

In [ ]:
#| label: tbl-nga-ledger
#| tbl-cap: "Nigeria, financial close 2014, USD debt service 100/yr with facility: the facility bridges every shortfall through the 2015-16 devaluation and the 2023 float; drawdowns never exceed one year's debt service."
# New (developed in session 2026-08-31): facility ledger on the realised path.
rho_l = 0.05
win = df[(df.iso3 == 'NGA') & (df.year >= 2014) & (df.year <= 2024)].sort_values('year').reset_index(drop=True)
S0, Q0 = win.exchange_rate.iloc[0], win.Q.iloc[0]
rows, bal = [], 0.0
for k in range(1, len(win)):
    r = win.iloc[k]
    g = r.Q / Q0 - 1
    flow = 100 * g                       # + = surplus to escrow, - = facility drawdown
    bal = bal * (1 + rho_l) + flow       # facility balance accrues at 5%
    rows.append({'year': int(r.year), 'fx': r.exchange_rate, 'Q_idx': 100 * r.Q / Q0,
                 'revenue_usd': 100 + flow, 'facility_flow': flow, 'facility_balance': bal})
ledger = pd.DataFrame(rows)
ledger.round(1)

In [ ]:
#| label: fig-nga-ledger
#| fig-cap: "Facility flows (bars) and running facility balance (line), Nigeria 2014 close."
bars = alt.Chart(ledger).mark_bar().encode(
    x=alt.X('year:O', title=None),
    y=alt.Y('facility_flow:Q', title='Facility flow (USD, per 100 debt service)'),
    color=alt.condition(alt.datum.facility_flow < 0, alt.value('crimson'), alt.value('steelblue')),
)
line = alt.Chart(ledger).mark_line(color='#333', point=True).encode(
    x='year:O',
    y=alt.Y('facility_balance:Q', title='Facility balance', axis=alt.Axis(orient='right')),
)
alt.layer(bars, line).resolve_scale(y='independent').properties(width=550, height=280)

## Nigeria 2014, refinanced: right in PV, dead in year two

The honest counterfactual: converting every naira payment back to USD at realised rates, the naira loan is ex-post *cheaper* in USD PV (~524 vs 772 - depreciation outran the priced differential). Yet it defaults immediately on DSCR (0.77 in 2015, 0.89 in 2016), because the erosion relief arrives after the front-loaded coupon bill - the tilt of @sec-hedging realised. The robust claims against the hedge are the natural-hedge decomposition, the tilt, and certain-vs-contingent cost - not expected-cost dominance.

In [ ]:
#| label: tbl-nga-refi
#| tbl-cap: "The 2014 naira refinance (13% coupon): USD-converted payments and DSCR against indexed revenues."
# New (session 2026-08-31): naira refinance at close, 10y annuity at 13%.
r_ngn, T_ref = 0.13, 10
af_usd10 = (1 - 1.05 ** -T_ref) / 0.05
af_ngn10 = (1 - (1 + r_ngn) ** -T_ref) / r_ngn
D_usd = 100 * af_usd10                       # 772.2
pay_ngn = D_usd * S0 / af_ngn10              # fixed naira annuity
CPI0 = win.cpi.iloc[0]
refi = pd.DataFrame({
    'year': win.year.iloc[1:].astype(int).values,
    'pay_usd': (pay_ngn / win.exchange_rate.iloc[1:]).values,
    'rev_ngn': (100 * S0 * win.cpi.iloc[1:] / CPI0).values,
})
refi['dscr_ngn'] = refi.rev_ngn / pay_ngn
pv_refi = float(np.sum(refi.pay_usd.values * 1.05 ** -np.arange(1, T_ref + 1)))
print(f"Naira annuity: N{pay_ngn:,.0f}/yr = {pay_ngn / S0:.1f} USD at close FX (vs 100 for the USD loan)")
print(f"PV of USD-converted naira payments @5%: {pv_refi:.1f}  vs USD loan {D_usd:.1f}")
refi.round(2)

## Kenya 2009: the certain coupon with no offset {#sec-kenya-refi}

The shilling refinance pays a ~7-point differential against realised depreciation of ~2.7%/yr: every single USD-converted payment exceeds the USD loan's 100, an ex-post overpayment of ~200bp of principal per year for a decade — while $Q$ never PV-net fell below any window start since 1995, so the Facility client paid ~19bp and never drew.


In [ ]:
#| label: tbl-ken-refi
#| tbl-cap: "Kenya, close 2009: KES refinance at 12% - USD-converted payments never fall below the USD loan's 100."
# New (session 2026-08-31).
r_kes = 0.12
wk = df[(df.iso3 == 'KEN') & (df.year >= 2009) & (df.year <= 2019)].sort_values('year').reset_index(drop=True)
S0k = wk.exchange_rate.iloc[0]
af_kes10 = (1 - (1 + r_kes) ** -10) / r_kes
pay_kes = D_usd * S0k / af_kes10
ken = pd.DataFrame({
    'year': wk.year.iloc[1:].astype(int).values,
    'pay_usd': (pay_kes / wk.exchange_rate.iloc[1:]).values,
})
pv_kes = float(np.sum(ken.pay_usd.values * 1.05 ** -np.arange(1, 11)))
extra_bp = (pv_kes - D_usd) / af_usd10 / D_usd * 1e4
print(f"PV of USD-converted KES payments @5%: {pv_kes:.1f} vs USD loan {D_usd:.1f}")
print(f"Ex-post overpayment: {pv_kes - D_usd:.1f} = flat {extra_bp:.0f}bp of principal/yr for 10 years")
ken.round(1)

## Scenario matrix

| | Certain annual cost | Nigeria 2014 path | Kenya 2009 path | Failure mode |
|---|---|---|---|---|
| USD debt, naked | 0 | technical default 2015-17 and 2023-24 | fine | real depreciation |
| **USD debt + facility** | **19bp pooled / <=97bp standalone** | every shortfall bridged, no default | premium paid, never drawn | none on any sample path |
| Refinance to LCU | full differential (contractual) | ex-post cheap in PV, **but DSCR default in year 1** | ex-post ~200bp/yr overpaid | tilt + rollover + market depth |
| Rolling NDF hedge | differential, repriced each roll | forward points explode after each deval | overpays like the refinance | rollover at the worst moment |

TODO: generate this table from the computations above rather than hard-coding.

In [ ]:
# TODO (new analysis - anti-hedge exhibit #2): distributions of minimum DSCR,
# not mean cost, across ALL historical windows for the three structures.
#
# Spec: for every rolling window and each structure
#   (a) naked USD:      DSCR_t = H_t/H_0
#   (b) USD + facility: DSCR_t = 1 by construction (cost = premium)
#   (c) LCU refinance:  DSCR_t = (CPI_t/CPI_0) / (annuity at coupon i_usd +
#                       trailing-5y inflation differential + spread; sensitivity
#                       over the spread assumption)
# compute min DSCR over the window and the number of years with DSCR < 1.
# One chart: three overlaid distributions of min-DSCR. The point: project
# finance prices the worst year - the hedge does not remove the left tail,
# it relabels it (real-depreciation risk -> disinflation risk + tilt).
pass

# Solvency: what capital the insurer actually needs {#sec-solvency}

Two distinct commitments. The *liquidity* commitment is bounded by one year of covered debt service: since $Q > 0$, the Facility can never advance more than one year's debt service in any year (worst observed single-year shortfall: ~75%, Nigeria 2024) — cheap for a AAA balance sheet such as the AfDB's to stand behind. The *solvency* capital must instead be sized off the terminal-loss tail of the (diversified) covered book, i.e. the upper tail of $L_i = 100(-V_i)^{+}$ (equivalently the left tail of $V_i$).


In [ ]:
#| label: tbl-tail
#| tbl-cap: "Terminal-loss tail across all windows, and the worst single-year drawdown."
# New (session 2026-08-31): per-window tail quantiles and worst annual shortfall.
rows = []
for _, w in residuals.iterrows():
    seg = df[(df.iso3 == w.iso3) & (df.year >= w.window_start) & (df.year <= w.window_end)]
    rows.append(seg.Q.min() / seg.Q.iloc[0] - 1)
tail = pd.Series(residuals['net_pv_by_debt_stock'])
print('terminal net PV / debt stock: q01 {:+.3f}  q05 {:+.3f}  q10 {:+.3f}  worst {:+.3f}'.format(
    tail.quantile(0.01), tail.quantile(0.05), tail.quantile(0.10), tail.min()))
print('worst single-year shortfall vs window start: {:+.3f}'.format(min(rows)))

In [ ]:
# TODO (new analysis - the EU grant number): portfolio VaR via the country-
# block bootstrap of @sec-variation.
#
# Spec: stylised covered book (e.g. EUR 1bn of covered annual debt service,
# staggered vintages across countries); reuse bootstrap_premium's resampling to
# build the portfolio terminal-loss distribution; report VaR_99 and VaR_99.5
# minus expected loss = risk buffer in EUR. That buffer is the grant-shaped
# quantity (candidate window: EFSD+ "Pricing Component Plus", open to Jan 2028).
# Also: time-profile of aggregate drawdowns across vintages (2015-16, 2020,
# 2023 cluster) -> when the buffer peaks, not just how big it is.
pass

# Policy design and conclusion {#sec-policy}

Under 30bp pooled - but adverse selection (the Nigerias buy, the Kenyas don't) unravels pooled pricing toward the ~150bp standalone worst case, arguing for embedding the cover in DFI lending terms or tiering premia by country (@fig-rate-by-country).

**Positioning in the current landscape.** The instrument is the complement, not the competitor, of the local-currency agenda: TCX and the Persaud partial FX guarantee operate on the *nominal* rate for lenders who convert; EcoInvest Brasil's FX liquidity line and this facility operate on *liquidity against the real component* for debt that stays in hard currency. The emerging hybrid PPA frameworks (Kenya, Nov 2025 / Jan 2026 regulations) explicitly retain a hard-currency component for international debt service - exactly the slice this facility makes safe. TODO: expand; cite TCX/AfDB (2026), IOB WP 2025.12, World Bank PRWP 10568, EFSD+.

**Caveats.** The post-1995 sample begins one year after the 1994 CFA devaluation (addressed by the start-year robustness of @sec-results); overlapping windows are handled by the country-block bootstrap.

In [ ]:
#| label: tbl-adverse-selection
#| tbl-cap: "Pooled premium excluding the k worst countries - the price of adverse selection."
# New: pooled premium as the worst countries opt out (or are tiered out).
worst_order = country_stats.sort_values('premium')['iso3'].tolist()  # most negative first
rows = []
for k in range(0, 6):
    keep = residuals[~residuals['iso3'].isin(worst_order[:k])]
    rows.append({'excluded (worst k)': k,
                 'countries': keep['iso3'].nunique(),
                 'premium (% debt stock)': -keep['loss'].mean() * 100,
                 'P(loss) (%)': (keep['loss'] < 0).mean() * 100})
pd.DataFrame(rows).round(3)

# Appendix {#sec-appendix .appendix}

- TODO: derivation of the $A$-identity and the bullet-vs-annuity spread correction, $A_{\rho+\Delta r,n} = A_{\rho,n}/(1+|\mathbb{E}[V_i]/A_{\rho,n}|)$.
- TODO: revenue-numeraire derivation — extra cost of local refinance $= rp$ when expectations are realised.
- Event-based robustness lives in `events_From_baseline.ipynb`.


In [ ]:
#| eval: false
#| label: fig-all-countries
#| fig-cap: "All-country panels: nominal rate vs Q (set eval: true to render; slow)."
# Ported from wdi_hybrid_analysis.ipynb, cell 11.
import math

countries = df[['iso3', 'country']].drop_duplicates().sort_values('country').reset_index(drop=True)

charts = []
for i, row in countries.iterrows():
    iso3, country_name = row['iso3'], row['country']
    cdata = df[(df['iso3'] == iso3) & (df['year'] >= start_year)].dropna(subset=['usd_per_lcu', 'Q'])
    if cdata.empty:
        continue

    legend_setting = alt.Legend(title="Series", orient="bottom") if i % 2 == 0 else None

    left_chart = alt.Chart(cdata).mark_line(point=True).encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('usd_per_lcu:Q', title='USD per LCU'),
        color=alt.Color('Metric:N', legend=legend_setting),
        tooltip=['year', 'usd_per_lcu']
    ).transform_calculate(Metric="'usd_per_lcu'")

    right_chart = alt.Chart(cdata).mark_line(point=True, color='orange').encode(
        x=alt.X('year:O', title='Year'),
        y=alt.Y('Q:Q', title='Q = CPI / S  (USD per local basket)', axis=alt.Axis(orient='right')),
        color=alt.Color('Metric:N', legend=legend_setting),
        tooltip=['year', 'Q']
    ).transform_calculate(Metric="'Q'")

    base = alt.layer(left_chart, right_chart).resolve_scale(y='independent').properties(
        title=country_name,
        width=400,
        height=300
    )
    charts.append(base)

def chart_list_to_rows_of_two(chart_list):
    rows = math.ceil(len(chart_list) / 2)
    chart_rows = []
    for i in range(rows):
        charts_in_row = chart_list[i*2 : i*2 + 2]
        if len(charts_in_row) == 2:
            r = alt.hconcat(*charts_in_row).resolve_scale(color='independent')
        else:
            r = charts_in_row[0]
        chart_rows.append(r)
    return alt.vconcat(*chart_rows)

final_layout = chart_list_to_rows_of_two(charts)
final_layout